In [ ]:
# Keep-alive script to prevent Colab from disconnecting
import IPython
from google.colab import output

display(IPython.display.Javascript('''
 function ClickConnect(){
   console.log("Attempting to keep session alive"); 
   document.querySelector("colab-connect-button").click()
 }
 setInterval(ClickConnect, 60000)
'''))

print("✓ Keep-alive script activated")

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully")

## Step 2: Install Required Libraries

In [ ]:
# Install YOLOv8 and dependencies
!pip install -q ultralytics opencv-python-headless pillow
print("✓ Libraries installed")

## Step 3: Set Up Paths

**IMPORTANT**: Update the `DATASET_PATH` below to point to your dataset folder in Google Drive!

In [ ]:
import os

# ⚠️ CHANGE THIS PATH to your dataset location in Google Drive
DATASET_PATH = "/content/drive/MyDrive/potholes_mini"

# Verify path exists
if not os.path.exists(DATASET_PATH):
    print(f"❌ ERROR: Dataset path not found: {DATASET_PATH}")
    print("Please upload your dataset to Google Drive and update DATASET_PATH")
else:
    print(f"✓ Dataset found at: {DATASET_PATH}")
    
# Create working directory
WORK_DIR = "/content/potholes_work"
os.makedirs(WORK_DIR, exist_ok=True)
print(f"✓ Working directory: {WORK_DIR}")

## Step 4: Define Class Names

For pothole detection, typically only 1 class is needed.

In [ ]:
# Pothole detection usually has just 1 class
# You can add more classes if needed (e.g., crack, bump, etc.)
CLASS_NAMES = [
    "pothole"
]

# If you have multiple road defect types, update like this:
# CLASS_NAMES = [
#     "pothole",
#     "crack",
#     "bump"
# ]

NUM_CLASSES = len(CLASS_NAMES)
print(f"✓ Number of classes: {NUM_CLASSES}")
print(f"✓ Classes: {CLASS_NAMES}")

## Step 5: Create YOLO Configuration File

In [ ]:
import yaml

# Create YOLO data.yaml configuration
data_config = {
    'path': DATASET_PATH,
    'train': 'images/train',
    'val': 'images/val',
    'nc': NUM_CLASSES,
    'names': CLASS_NAMES
}

config_path = os.path.join(WORK_DIR, 'data.yaml')
with open(config_path, 'w') as f:
    yaml.dump(data_config, f, sort_keys=False)

print(f"✓ Configuration saved to: {config_path}")
print("\nConfiguration:")
print(yaml.dump(data_config, sort_keys=False))

## Step 6: Verify Dataset Structure

In [ ]:
import glob

# Check train images and labels
train_images = glob.glob(os.path.join(DATASET_PATH, 'images/train/*'))
train_labels = glob.glob(os.path.join(DATASET_PATH, 'labels/train/*'))

# Check val images and labels
val_images = glob.glob(os.path.join(DATASET_PATH, 'images/val/*'))
val_labels = glob.glob(os.path.join(DATASET_PATH, 'labels/val/*'))

print("📊 Dataset Statistics:")
print(f"Train Images: {len(train_images)}")
print(f"Train Labels: {len(train_labels)}")
print(f"Val Images: {len(val_images)}")
print(f"Val Labels: {len(val_labels)}")

if len(train_images) == 0:
    print("\n❌ ERROR: No training images found!")
    print(f"Check: {os.path.join(DATASET_PATH, 'images/train/')}")
elif len(train_images) != len(train_labels):
    print(f"\n⚠️ WARNING: Mismatch between images ({len(train_images)}) and labels ({len(train_labels)})")
else:
    print("\n✓ Dataset structure looks good!")

## Step 7: Visualize Sample Images (Optional)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from PIL import Image

# Display first 3 training images
sample_images = train_images[:3]

fig, axes = plt.subplots(1, min(3, len(sample_images)), figsize=(15, 5))
if len(sample_images) == 1:
    axes = [axes]

for idx, img_path in enumerate(sample_images):
    img = Image.open(img_path)
    axes[idx].imshow(img)
    axes[idx].set_title(f"Sample {idx+1}")
    axes[idx].axis('off')

plt.tight_layout()
plt.show()
print("✓ Sample images displayed")

## Step 8: Train YOLOv8 Model

Training with **small dataset optimizations**:
- Small batch size (8)
- Fewer epochs (30-50)
- Nano model (yolov8n) for faster training
- Data augmentation enabled

In [ ]:
from ultralytics import YOLO

# Initialize YOLOv8 nano model (fastest)
model = YOLO('yolov8n.pt')

# Training parameters optimized for small dataset
results = model.train(
    data=config_path,
    epochs=30,              # Fewer epochs for demo (adjust 30-50)
    imgsz=640,              # Image size
    batch=8,                # Small batch size
    name='pothole_mini',
    project=WORK_DIR,
    cache=False,            # Don't cache (small dataset)
    workers=2,              # Fewer workers
    device=0,               # Use GPU
    patience=10,            # Early stopping patience
    save=True,
    plots=True,
    # Data augmentation (helps with small datasets)
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    # Pothole-specific augmentations
    perspective=0.0001,     # Slight perspective changes
    flipud=0.2              # Vertical flip (useful for road images)
)

print("\n✓ Training complete!")

## Step 9: Evaluate Model Performance

In [ ]:
# Validate the model
metrics = model.val()

print("\n📊 Model Performance:")
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

## Step 10: Test Model on Sample Image

In [ ]:
# Test on first validation image
if len(val_images) > 0:
    test_image = val_images[0]
    results = model.predict(test_image, conf=0.25)
    
    # Display results
    result_plot = results[0].plot()
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(result_plot, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('Model Prediction on Validation Image')
    plt.show()
    print("✓ Prediction displayed")
else:
    print("⚠️ No validation images available for testing")

## Step 11: Export to TFLite Format

In [ ]:
# Find the best model weights
best_model_path = os.path.join(WORK_DIR, 'pothole_mini', 'weights', 'best.pt')

if os.path.exists(best_model_path):
    # Load best model
    best_model = YOLO(best_model_path)
    
    # Export to TFLite with INT8 quantization
    print("Exporting to TFLite format...")
    tflite_path = best_model.export(
        format='tflite',
        imgsz=640,
        int8=True,
        data=config_path
    )
    print(f"✓ TFLite model exported: {tflite_path}")
else:
    print(f"❌ Best model not found at: {best_model_path}")

## Step 12: Copy Model to Google Drive

In [ ]:
import shutil

# Define output path in Google Drive
drive_output_path = "/content/drive/MyDrive/RoadAI_Models/patholes_mini.tflite"

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(drive_output_path), exist_ok=True)

# Find the exported TFLite file
tflite_files = glob.glob(os.path.join(WORK_DIR, 'pothole_mini', 'weights', '*_saved_model', '*_int8.tflite'))

if len(tflite_files) > 0:
    source_tflite = tflite_files[0]
    shutil.copy(source_tflite, drive_output_path)
    
    # Get file size
    file_size = os.path.getsize(drive_output_path) / (1024 * 1024)
    
    print(f"✓ Model saved to Google Drive!")
    print(f"Location: {drive_output_path}")
    print(f"Size: {file_size:.2f} MB")
    print(f"\n📱 Next Steps:")
    print(f"1. Download this file from Google Drive")
    print(f"2. Rename it to: patholes.tflite")
    print(f"3. Place in: app/src/main/assets/models/")
    print(f"4. Rebuild app: .\\gradlew assembleDebug")
else:
    print("❌ TFLite file not found!")
    print(f"Search path: {os.path.join(WORK_DIR, 'pothole_mini', 'weights')}")

## Step 13: Display Training Results

In [ ]:
# Display training curves
results_img = os.path.join(WORK_DIR, 'pothole_mini', 'results.png')

if os.path.exists(results_img):
    img = Image.open(results_img)
    plt.figure(figsize=(16, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Results')
    plt.show()
    print("✓ Training curves displayed")
else:
    print("⚠️ Results image not found")

## 📝 Summary

### What This Notebook Does:
1. ✓ Mounts Google Drive
2. ✓ Loads your custom mini pothole dataset
3. ✓ Trains YOLOv8 model (optimized for small data)
4. ✓ Evaluates performance
5. ✓ Exports to TFLite format
6. ✓ Saves model to Google Drive

### Important Notes:
- **Dataset Size**: Recommended 20-50 images for quick demo
- **Training Time**: ~10-20 minutes with small dataset
- **Accuracy**: Limited due to small dataset (good for demo only)
- **Production Use**: Train with larger dataset (300+ images) for real deployment

### Model Location:
- **Google Drive**: `/content/drive/MyDrive/RoadAI_Models/patholes_mini.tflite`
- **Download and rename**: `patholes.tflite`
- **Place in app**: `app/src/main/assets/models/`

### Tips for Better Results:
- Use diverse images (different lighting, angles, road types)
- Include various pothole sizes and severities
- Label carefully with accurate bounding boxes
- Consider 70/30 train/val split for small datasets